# ODI to Databricks Migration

**Source File:** TARGET
**Conversion Timestamp:** 2024-07-30 12:00:00 UTC

This notebook contains the converted Spark SQL logic from the original ODI session tasks. It performs an insert into the `workspace.hr.trg_loc` table.

In [ ]:
import dbutils

dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT ${ETL_PROC_WID} AS etl_proc_wid;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("""
  SELECT
    v_etl_job_type.etl_job_type,
    v_datasource_num_id.datasource_num_id,
    v_etl_proc_wid.etl_proc_wid,
    v_odi_sess_no.odi_sess_no
  FROM v_etl_job_type, v_datasource_num_id, v_etl_proc_wid, v_odi_sess_no
"""))

# Target Load

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT 
  INTO workspace.hr.trg_loc
  (
    LOCATION_ID ,
    STREET_ADDRESS ,
    POSTAL_CODE ,
    CITY ,
    STATE_PROVINCE ,
    COUNTRY_ID 
  ) 
SELECT 
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID  
FROM 
  workspace.hr.locations AS LOCATIONS;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS target_row_count FROM workspace.hr.trg_loc;

# Cleanup

# Conversion Notes

No specific manual actions are required for this conversion. The original ODI SQL was a simple `INSERT SELECT` statement without complex features like `MERGE`, staging tables, or error handling.

**Key Changes:**
- Removed Oracle hint `/*+ APPEND PARALLEL */` as it's not applicable in Spark SQL / Delta Lake.
- Converted Oracle schema and table names (`HR.TRG_LOC`, `HR.LOCATIONS`) to Databricks Delta Lake format (`workspace.hr.trg_loc`, `workspace.hr.locations`).
- The `SCEN_TASK_NO` markers `{10}`, `{20}`, `{30}` were consolidated as comments preceding the `INSERT` statement, as they represent sequential execution points rather than distinct operational tasks in this simplified flow.